# CNN + LSTM + Projector + Decoder — Attention Placement Experiments

## Kiến trúc tổng thể

```
  Ảnh  ──▶  CNN Encoder  ──▶  v_img [B, 32]
                                     │
                              ┌──────┤  (tuỳ chọn: Projector)
                              │      │
                              ▼      ▼
                          Projector  Projector
                           (32→P)    (16→P)
                              │      │
                              ▼      ▼
                              CONCAT  ──▶  DECODER  ──▶  ŷ  ──▶  LOSS
                              │      │
                              ▲      ▲
                              │      │
  Text ──▶ LSTM Encoder ──▶  v_txt [B, 16]
```

## Thí nghiệm

| Case | CNN Attn | LSTM Attn | Decoder Attn | Projector | Mô tả |
|---|:---:|:---:|:---:|:---:|---|
| 1. Baseline | ✗ | ✗ | ✗ | ✗ | Không có gì thêm |
| 2. CNN-only | ✓ | ✗ | ✗ | ✗ | Chỉ CNN có Channel Attention |
| 3. LSTM-only | ✗ | ✓ | ✗ | ✗ | Chỉ LSTM có Temporal Attention |
| 4. Decoder-only | ✗ | ✗ | ✓ | ✗ | Chỉ Decoder có Gated Attention |
| 5. Full Attn | ✓ | ✓ | ✓ | ✗ | Cả 3 vị trí có Attention |
| 6. Projector only | ✗ | ✗ | ✗ | ✓ | Chỉ có Projector, không Attention |
| 7. Full Attn + Proj | ✓ | ✓ | ✓ | ✓ | Đầy đủ: Attention + Projector |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

---
## 1. Dữ liệu

In [ ]:
!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d puneet6060/intel-image-classification
!unzip -q intel-image-classification.zip

In [ ]:
transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder("seg_train/seg_train", transform=transform)
test_dataset  = datasets.ImageFolder("seg_test/seg_test",  transform=transform)

In [ ]:
captions = {
    0: ["modern buildings in the city",
        "urban skyline with tall skyscrapers",
        "city street lined with buildings"],
    1: ["dense green forest",
        "thick forest full of trees",
        "green woodland scenery"],
    2: ["beautiful glacier covered with ice",
        "icy glacier landscape with snow",
        "frozen glacier under the sky"],
    3: ["high mountain landscape",
        "tall mountain peak scenery",
        "mountain range under the sky"],
    4: ["blue sea with water",
        "ocean waves near the shore",
        "sea view with blue water"],
    5: ["urban street with roads",
        "city road full of traffic",
        "street with parked cars"],
}
NUM_CLASSES = len(captions)

In [ ]:
words = set()
for variants in captions.values():
    for s in variants:
        words.update(s.split())

word2idx = {w: i + 1 for i, w in enumerate(sorted(words))}
word2idx["<pad>"] = 0
VOCAB_SIZE = len(word2idx)

max_len = max(len(s.split()) for vs in captions.values() for s in vs)

In [ ]:
class MultiModalDataset(Dataset):
    def __init__(self, base_dataset, train_mode=True):
        self.base = base_dataset
        self.train_mode = train_mode

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        image, label = self.base[idx]
        variants = captions[label]
        sentence = random.choice(variants) if self.train_mode else variants[0]
        tokens = [word2idx[w] for w in sentence.split()]
        tokens = tokens[:max_len] + [0] * (max_len - len(tokens))
        return image, torch.tensor(tokens), label

trainloader = DataLoader(MultiModalDataset(train_dataset, True),  batch_size=64, shuffle=True)
testloader  = DataLoader(MultiModalDataset(test_dataset,  False), batch_size=64, shuffle=False)

---
## 2. Các module Attention

### 2.1 Channel Attention (SE-Block) — dùng cho CNN

```
Feature Map [B, C, H, W]
  → Global Avg Pool       → [B, C]
  → FC → ReLU → FC → σ   → [B, C]
  → Scale từng kênh       → [B, C, H, W]
```

In [ ]:
class ChannelAttention(nn.Module):
    """SE-Block: hoc trong so quan trong cho tung kenh CNN."""

    def __init__(self, channels, reduction=4):
        super().__init__()
        mid = max(channels // reduction, 1)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid),
            nn.ReLU(),
            nn.Linear(mid, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        squeeze = x.mean(dim=[2, 3])                   # [B, C]
        weights = self.fc(squeeze).view(b, c, 1, 1)    # [B, C, 1, 1]
        return x * weights

### 2.2 Temporal Attention — dùng cho LSTM

```
LSTM outputs [B, T, H]
  → Linear(H, 1) → scores [B, T]
  → Masked Softmax → weights [B, T]
  → Weighted Sum → context [B, H]
```

In [ ]:
class TemporalAttention(nn.Module):
    """Attention tren cac buoc thoi gian cua LSTM."""

    def __init__(self, hidden_dim):
        super().__init__()
        self.score_fn = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs, pad_mask=None):
        scores = self.score_fn(lstm_outputs).squeeze(-1)     # [B, T]
        if pad_mask is not None:
            scores = scores.masked_fill(pad_mask, float('-inf'))
        weights = torch.softmax(scores, dim=1)                # [B, T]
        context = (lstm_outputs * weights.unsqueeze(-1)).sum(dim=1)  # [B, H]
        return context, weights

### 2.3 Gated Attention — dùng cho Decoder

```
Fused [B, D]
  → Linear(D, D) → Sigmoid → gate [B, D]
  → fused * gate → [B, D]
```

In [ ]:
class GatedAttention(nn.Module):
    """Gated Attention: hoc cong loc thong tin tren fused features."""

    def __init__(self, dim):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.gate(x)

---
## 3. Projector (MLP Mapper)

### Vì sao cần Projector?

CNN và LSTM tạo ra các vector đặc trưng ở **không gian latent khác nhau**:
- `v_img` ∈ ℝ³² — không gian thị giác (biểu diễn hình dạng, màu sắc, kết cấu)
- `v_txt` ∈ ℝ¹⁶ — không gian ngữ nghĩa (biểu diễn ý nghĩa từ, cú pháp)

Nếu **concat trực tiếp** `[v_img; v_txt]`, Decoder phải tự học cách "dịch" giữa 2 không gian → khó hơn.

**Projector** chiếu cả 2 vào một **không gian chung (shared latent space)** trước khi ghép, giúp fusion hiệu quả hơn.

```
  v_img [B, 32]  ──▶  ImgProjector (32→P)  ──▶  z_img [B, P]  ──┐
                                                                  ├──▶  CONCAT  ──▶  [B, 2P]
  v_txt [B, 16]  ──▶  TxtProjector (16→P)  ──▶  z_txt [B, P]  ──┘
```

### Tương tự Medical-VQA baseline

Trong bài Medical-VQA (Open_Ended_VQA.ipynb), MLP Mapper làm đúng vai trò này:
- CLIP image encoder xuất vector 512-dim
- GPT-2 decoder mong đợi embedding 768-dim
- **MLP Mapper**: `Linear(512→768) → ReLU → Linear(768→768)` — chiếu ảnh sang không gian embedding của text

Projector ở đây cũng tương tự, nhưng áp dụng cho **cả 2 modality** để đưa vào shared space.

In [ ]:
class Projector(nn.Module):
    """MLP Projector: chieu dac trung tu khong gian rieng
    sang khong gian chung (shared latent space).

    Cau truc: Linear -> GELU -> Linear -> LayerNorm

    - Lop Linear dau: hoc phep bien doi phi tuyen tu input_dim -> proj_dim
    - GELU: activation mem hon ReLU, pho bien trong Transformer
    - Lop Linear sau: tinh chinh bieu dien trong khong gian moi
    - LayerNorm: on dinh scale, giup training hoi tu nhanh hon
    """

    def __init__(self, input_dim, proj_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, proj_dim),
            nn.GELU(),
            nn.Linear(proj_dim, proj_dim),
            nn.LayerNorm(proj_dim),
        )

    def forward(self, x):
        return self.net(x)

# Minh hoa kieu du lieu qua Projector:
# img_proj = Projector(32, 32)  #  v_img [B, 32] -> z_img [B, 32]
# txt_proj = Projector(16, 32)  #  v_txt [B, 16] -> z_txt [B, 32]  ← chiều thay đổi!
# => concat: [B, 64]  (thay vì [B, 48] khi không có projector)

---
## 4. Kiến trúc mô hình

### 4.1 CNN Encoder (Image)

In [ ]:
class ImageEncoder(nn.Module):

    def __init__(self, use_attention=False):
        super().__init__()
        self.use_attention = use_attention

        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
        )

        if use_attention:
            self.attention = ChannelAttention(32)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.norm = nn.LayerNorm(32)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        if self.use_attention:
            x = self.attention(x)
        x = self.pool(x)
        x = x.flatten(1)
        return self.norm(x)                  # [B, 32]

### 4.2 LSTM Encoder (Text)

In [ ]:
class TextEncoderLSTM(nn.Module):

    def __init__(self, use_attention=False):
        super().__init__()
        self.use_attention = use_attention

        self.embedding = nn.Embedding(VOCAB_SIZE, 16, padding_idx=0)
        self.lstm = nn.LSTM(input_size=16, hidden_size=16,
                           num_layers=1, batch_first=True)

        if use_attention:
            self.attention = TemporalAttention(hidden_dim=16)

        self.norm = nn.LayerNorm(16)

    def forward(self, x):
        pad_mask = (x == 0)
        emb = self.embedding(x)
        lstm_out, (h_n, _) = self.lstm(emb)

        if self.use_attention:
            out, _ = self.attention(lstm_out, pad_mask)
        else:
            out = h_n.squeeze(0)

        return self.norm(out)                # [B, 16]

### 4.3 Decoder

In [ ]:
class Decoder(nn.Module):

    def __init__(self, input_dim, num_classes, use_attention=False):
        super().__init__()
        self.use_attention = use_attention

        if use_attention:
            self.attention = GatedAttention(input_dim)

        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        if self.use_attention:
            x = self.attention(x)
        return self.classifier(x)

### 4.4 Fusion Model — Kết hợp tất cả

4 cờ tuỳ chọn:
- `cnn_attention` — bật Channel Attention trong CNN
- `lstm_attention` — bật Temporal Attention trong LSTM
- `decoder_attention` — bật Gated Attention trong Decoder
- `use_projector` — bật Projector giữa Encoder và Decoder

```
KHÔNG có Projector:                   CÓ Projector:
  v_img [32] ─┐                        v_img [32] → Proj(32→P) → z_img [P] ─┐
               ├→ concat [48] → Dec                                           ├→ concat [2P] → Dec
  v_txt [16] ─┘                        v_txt [16] → Proj(16→P) → z_txt [P] ─┘
```

In [ ]:
PROJ_DIM = 32   # Chieu cua khong gian chung

class FusionModel(nn.Module):

    def __init__(self, cnn_attention=False, lstm_attention=False,
                 decoder_attention=False, use_projector=False):
        super().__init__()
        self.use_projector = use_projector

        # --- Encoders ---
        self.image = ImageEncoder(use_attention=cnn_attention)
        self.text  = TextEncoderLSTM(use_attention=lstm_attention)

        # --- Projector (tuy chon) ---
        if use_projector:
            self.img_proj = Projector(input_dim=32, proj_dim=PROJ_DIM)  # 32 → 32
            self.txt_proj = Projector(input_dim=16, proj_dim=PROJ_DIM)  # 16 → 32  ← thay doi chieu!
            fused_dim = PROJ_DIM * 2   # 64
        else:
            fused_dim = 32 + 16        # 48

        # --- Decoder ---
        self.decoder = Decoder(input_dim=fused_dim, num_classes=NUM_CLASSES,
                              use_attention=decoder_attention)

    def forward(self, image, text):
        v_img = self.image(image)                        # [B, 32]
        v_txt = self.text(text)                          # [B, 16]

        if self.use_projector:
            v_img = self.img_proj(v_img)                 # [B, PROJ_DIM]
            v_txt = self.txt_proj(v_txt)                 # [B, PROJ_DIM]

        fused  = torch.cat([v_img, v_txt], dim=1)        # [B, 48] hoac [B, 64]
        logits = self.decoder(fused)
        return logits

---
## 5. Hàm Train & Evaluate

In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, texts, labels in loader:
            images, texts, labels = images.to(device), texts.to(device), labels.to(device)
            outputs = model(images, texts)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total


def train_one_case(name, model, optimizer, criterion, device, epochs):
    model.to(device)
    history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, texts, labels in trainloader:
            images, texts, labels = images.to(device), texts.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images, texts)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / len(trainloader)
        train_acc  = correct / total
        test_loss, test_acc = evaluate(model, testloader, criterion, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)

        print(f"[{name}] Epoch {epoch+1}/{epochs} | "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
              f"test_loss={test_loss:.4f} test_acc={test_acc:.4f}")

    return history

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 8
criterion = nn.CrossEntropyLoss()
print(f"Device: {device}")

---
## 6. Thực nghiệm

### Case 1: Baseline — Không Attention, không Projector

In [ ]:
torch.manual_seed(SEED)
m1 = FusionModel(cnn_attention=False, lstm_attention=False,
                 decoder_attention=False, use_projector=False)
opt1 = torch.optim.Adam(m1.parameters(), lr=1e-3)
h1 = train_one_case("Baseline", m1, opt1, criterion, device, EPOCHS)

### Case 2: Chỉ CNN có Attention (Channel Attention / SE-Block)

In [ ]:
torch.manual_seed(SEED)
m2 = FusionModel(cnn_attention=True, lstm_attention=False,
                 decoder_attention=False, use_projector=False)
opt2 = torch.optim.Adam(m2.parameters(), lr=1e-3)
h2 = train_one_case("CNN attn", m2, opt2, criterion, device, EPOCHS)

### Case 3: Chỉ LSTM có Attention (Temporal Attention)

In [ ]:
torch.manual_seed(SEED)
m3 = FusionModel(cnn_attention=False, lstm_attention=True,
                 decoder_attention=False, use_projector=False)
opt3 = torch.optim.Adam(m3.parameters(), lr=1e-3)
h3 = train_one_case("LSTM attn", m3, opt3, criterion, device, EPOCHS)

### Case 4: Chỉ Decoder có Attention (Gated Attention)

In [ ]:
torch.manual_seed(SEED)
m4 = FusionModel(cnn_attention=False, lstm_attention=False,
                 decoder_attention=True, use_projector=False)
opt4 = torch.optim.Adam(m4.parameters(), lr=1e-3)
h4 = train_one_case("Decoder attn", m4, opt4, criterion, device, EPOCHS)

### Case 5: Full Attention (CNN + LSTM + Decoder)

In [ ]:
torch.manual_seed(SEED)
m5 = FusionModel(cnn_attention=True, lstm_attention=True,
                 decoder_attention=True, use_projector=False)
opt5 = torch.optim.Adam(m5.parameters(), lr=1e-3)
h5 = train_one_case("Full attn", m5, opt5, criterion, device, EPOCHS)

### Case 6: Chỉ Projector — không Attention

So sánh tác dụng **thuần tuý** của Projector: chiếu img (32→32) và txt (16→32) vào shared space.

In [ ]:
torch.manual_seed(SEED)
m6 = FusionModel(cnn_attention=False, lstm_attention=False,
                 decoder_attention=False, use_projector=True)
opt6 = torch.optim.Adam(m6.parameters(), lr=1e-3)
h6 = train_one_case("Projector only", m6, opt6, criterion, device, EPOCHS)

### Case 7: Full Attention + Projector — Đầy đủ nhất

Kết hợp tất cả: Channel Attention + Temporal Attention + Projector + Gated Attention.

In [ ]:
torch.manual_seed(SEED)
m7 = FusionModel(cnn_attention=True, lstm_attention=True,
                 decoder_attention=True, use_projector=True)
opt7 = torch.optim.Adam(m7.parameters(), lr=1e-3)
h7 = train_one_case("Full + Projector", m7, opt7, criterion, device, EPOCHS)

---
## 7. So sánh kết quả

### 7.1 Bảng tổng hợp

In [ ]:
all_results = {
    "1. Baseline":           h1,
    "2. CNN attn":           h2,
    "3. LSTM attn":          h3,
    "4. Decoder attn":       h4,
    "5. Full attn":          h5,
    "6. Projector only":     h6,
    "7. Full + Projector":   h7,
}

flags_list = [
    # CNN  LSTM  Dec  Proj
    ("✗", "✗", "✗", "✗"),
    ("✓", "✗", "✗", "✗"),
    ("✗", "✓", "✗", "✗"),
    ("✗", "✗", "✓", "✗"),
    ("✓", "✓", "✓", "✗"),
    ("✗", "✗", "✗", "✓"),
    ("✓", "✓", "✓", "✓"),
]

print("=" * 92)
print(f"{'Case':25}{'CNN':>5}{'LSTM':>6}{'Dec':>5}{'Proj':>6}{'Test Loss':>14}{'Test Acc':>14}")
print("=" * 92)
for (name, h), fl in zip(all_results.items(), flags_list):
    fc, fl2, fd, fp = fl
    print(f"{name:25}{fc:>5}{fl2:>6}{fd:>5}{fp:>6}"
          f"{h['test_loss'][-1]:>14.4f}{h['test_acc'][-1]:>14.4f}")
print("=" * 92)

### 7.2 Số lượng tham số

In [ ]:
all_models = {
    "1. Baseline":         m1,
    "2. CNN attn":         m2,
    "3. LSTM attn":        m3,
    "4. Decoder attn":     m4,
    "5. Full attn":        m5,
    "6. Projector only":   m6,
    "7. Full + Projector": m7,
}

print(f"{'Model':25}{'Image':>10}{'Text':>10}{'Proj':>10}{'Decoder':>10}{'Total':>10}")
print("-" * 75)
for name, mdl in all_models.items():
    n_img = sum(p.numel() for p in mdl.image.parameters())
    n_txt = sum(p.numel() for p in mdl.text.parameters())
    n_proj = 0
    if mdl.use_projector:
        n_proj = (sum(p.numel() for p in mdl.img_proj.parameters())
                  + sum(p.numel() for p in mdl.txt_proj.parameters()))
    n_dec = sum(p.numel() for p in mdl.decoder.parameters())
    n_all = sum(p.numel() for p in mdl.parameters())
    print(f"{name:25}{n_img:>10,}{n_txt:>10,}{n_proj:>10,}{n_dec:>10,}{n_all:>10,}")

### 7.3 Biểu đồ Test Loss & Test Accuracy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6))

palette = ["#636363", "#e6550d", "#3182bd", "#31a354",
           "#756bb1", "#e7298a", "#d95f02"]
styles  = ["-", "-", "-", "-", "-", "--", "--"]

for i, (name, h) in enumerate(all_results.items()):
    c, ls = palette[i], styles[i]
    axes[0].plot(h["test_loss"], label=name, color=c, linewidth=2,
                linestyle=ls, marker='o', markersize=4)
    axes[1].plot(h["test_acc"],  label=name, color=c, linewidth=2,
                linestyle=ls, marker='o', markersize=4)

axes[0].set_title("Test Loss", fontsize=14, fontweight='bold')
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend(fontsize=8);  axes[0].grid(True, alpha=0.3)

axes[1].set_title("Test Accuracy", fontsize=14, fontweight='bold')
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].legend(fontsize=8);  axes[1].grid(True, alpha=0.3)

plt.suptitle("Attention & Projector Placement Comparison",
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 7.4 Train vs Test Loss — Phát hiện Overfitting

In [ ]:
n_cases = len(all_results)
ncols = 4
nrows = math.ceil(n_cases / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes_flat = axes.flatten()

for i, (name, h) in enumerate(all_results.items()):
    ax = axes_flat[i]
    ep = range(1, EPOCHS + 1)
    ax.plot(ep, h["train_loss"], label="Train", color="#e6550d", linewidth=2)
    ax.plot(ep, h["test_loss"],  label="Test",  color="#3182bd", linewidth=2)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.legend(fontsize=8);  ax.grid(True, alpha=0.3)

for j in range(i + 1, len(axes_flat)):
    axes_flat[j].axis('off')

plt.suptitle("Train vs Test Loss", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 7.5 Bar Chart — So sánh Accuracy cuối cùng

In [ ]:
names = list(all_results.keys())
accs  = [h["test_acc"][-1] for h in all_results.values()]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(names, accs, color=palette, edgecolor='white', linewidth=1.5)

for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{acc:.4f}", ha='center', va='bottom', fontweight='bold', fontsize=10)

ax.set_ylabel("Test Accuracy", fontsize=12)
ax.set_title("Final Test Accuracy — All Cases", fontsize=14, fontweight='bold')
ax.set_ylim(0, max(accs) * 1.15)
plt.xticks(rotation=25, ha='right')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

### 7.6 In cấu trúc tham số chi tiết

In [ ]:
for name, mdl in all_models.items():
    print(f"===== {name} =====")
    for pname, param in mdl.named_parameters():
        print(f"  {pname:45} {str(list(param.shape)):20}")
    total = sum(p.numel() for p in mdl.parameters())
    print(f"  {'TOTAL':45} {total:>10,}")
    print()

---
## 8. Tóm tắt & Phân tích

### Vai trò từng thành phần

| Thành phần | Vị trí | Chức năng |
|---|---|---|
| **Channel Attention** (SE-Block) | CNN Encoder | Re-weight **kênh** feature map — tập trung vào đặc trưng thị giác quan trọng |
| **Temporal Attention** | LSTM Encoder | Re-weight **từng từ** — tập trung vào từ khoá quan trọng trong câu |
| **Projector** (MLP Mapper) | Giữa Encoder và Decoder | Chiếu img/txt vào **shared latent space** — căn chỉnh 2 modality trước khi fusion |
| **Gated Attention** | Decoder | **Lọc** thông tin thừa trong fused vector trước khi phân loại |

### Luồng dữ liệu đầy đủ (Case 7)

```
Ảnh  → Conv1 → Conv2 → [Channel Attention] → AvgPool → v_img [32]
                                                            │
                                                    [ImgProjector 32→32]
                                                            │
                                                            ├─── CONCAT ─── [64]
                                                            │              │
                                                    [TxtProjector 16→32]  [Gated Attention]
                                                            │              │
Text → Embed → LSTM → [Temporal Attention] → v_txt [16]   FC(64→32→6) → logits
```

### So sánh với Medical-VQA baseline

| | Notebook này | Medical-VQA (Open_Ended_VQA) |
|---|---|---|
| **Vision Encoder** | CNN (2 conv layers) | PubMed-CLIP ViT |
| **Text Encoder** | Embedding + LSTM | GPT-2 Embedding |
| **Projector** | `Projector(input_dim, proj_dim)` | `MLP(512→768→768)` |
| **Fusion** | Concat | Visual Prefix Injection |
| **Decoder** | FC + Gated Attention | GPT-2 Transformer Decoder |
| **Output** | Class logits (classification) | Token-by-token (generation) |